In [1]:
import pandas as pd
import dataclasses
from models import Ride, ride_from_row, ride_serializer

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"

In [3]:
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'passenger_count', 'tip_amount']
df = pd.read_parquet(url, columns=columns).head(1000)

In [4]:
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,lpep_pickup_datetime,lpep_dropoff_datetime,passenger_count,tip_amount
0,247,69,0.70,10.00,2025-10-01 00:21:47,2025-10-01 00:24:37,1.0,1.70
1,66,25,1.61,16.68,2025-10-01 00:14:03,2025-10-01 00:24:14,1.0,2.78
2,244,244,0.00,13.20,2025-10-01 00:16:44,2025-10-01 00:16:47,1.0,2.20
3,95,170,10.37,67.85,2025-10-01 00:07:36,2025-10-01 00:32:14,1.0,11.31
4,82,138,4.07,34.12,2025-09-30 21:10:29,2025-09-30 21:22:30,1.0,6.82


In [ ]:
#checking the first row of the dataframe for testing purposes
row = df.iloc[0]
row

In [ ]:
#checking the timestamp of the pickup datetime for testing purposes
row.lpep_pickup_datetime.timestamp()

In [ ]:
#creating an object for testing purposes
ride = ride_from_row(df.iloc[0])
ride
#ouput: Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, total_amount=10.0, 
#lpep_pickup_datetime='1759278107000', lpep_dropoff_datetime='1759278277000', passenger_count=1, tip_amount=1.7)

In [5]:
#creating an object manually for testing purposes
ride = Ride(
    PULocationID=1,
    DOLocationID=10,
    trip_distance=10,
    total_amount=20,
    lpep_pickup_datetime=str(1761956005000),
    lpep_dropoff_datetime=str(1861959605000),
    passenger_count=2,
    tip_amount=3
)

In [6]:
#creating a kafka producer
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)


In [7]:
ride_serializer(ride)

b'{"PULocationID": 1, "DOLocationID": 10, "trip_distance": 10, "total_amount": 20, "lpep_pickup_datetime": "1761956005000", "lpep_dropoff_datetime": "1861959605000", "passenger_count": 2, "tip_amount": 3}'

In [8]:
#defining a topic

topic_name = 'green-trips'

In [ ]:
#sending one topic for test purposes.

#producer.send(topic_name, value=dataclasses.asdict(ride))
producer.send(topic_name, value=ride)
producer.flush()

In [10]:
#sending all topics at the same time
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, total_amount=10.0, lpep_pickup_datetime='1759278107000', lpep_dropoff_datetime='1759278277000', passenger_count=1, tip_amount=1.7)
Sent: Ride(PULocationID=66, DOLocationID=25, trip_distance=1.61, total_amount=16.68, lpep_pickup_datetime='1759277643000', lpep_dropoff_datetime='1759278254000', passenger_count=1, tip_amount=2.78)
Sent: Ride(PULocationID=244, DOLocationID=244, trip_distance=0.0, total_amount=13.2, lpep_pickup_datetime='1759277804000', lpep_dropoff_datetime='1759277807000', passenger_count=1, tip_amount=2.2)
Sent: Ride(PULocationID=95, DOLocationID=170, trip_distance=10.37, total_amount=67.85, lpep_pickup_datetime='1759277256000', lpep_dropoff_datetime='1759278734000', passenger_count=1, tip_amount=11.31)
Sent: Ride(PULocationID=82, DOLocationID=138, trip_distance=4.07, total_amount=34.12, lpep_pickup_datetime='1759266629000', lpep_dropoff_datetime='1759267350000', passenger_count=1, tip_amount=6.82)
Sent: Rid